# Week 4 — Lab
## Tracking a PyTorch training loop with MLflow, then sweeping with Optuna

In this lab we:

1. Wrap a small CNN training loop on **FashionMNIST** with MLflow logging.
2. Run a 20-trial **Optuna** sweep over learning rate, weight decay, batch size, and
   dropout.
3. Pull the sweep dataframe out of the MLflow store and reproduce the four
   sweep-analysis plots **offline in Matplotlib / Plotly** — the same plots the
   trackers' web UIs show.
4. Log the best checkpoint as an artefact and demonstrate "load from run id".

The lab runs end-to-end on CPU in roughly 5–10 minutes, faster with a GPU. Everything is
local: MLflow writes to `./mlruns`, no account or network needed.


In [ ]:
# Setup
import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms as T
import mlflow
import optuna

plt.style.use("../../assets/mplstyle/course.mplstyle")
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

# MLflow writes to a local store next to the notebook
TRACKING = Path("./mlruns").resolve().as_uri()
mlflow.set_tracking_uri(TRACKING)
mlflow.set_experiment("fashionmnist-cnn")
print(f"MLflow tracking URI: {TRACKING}")


## 1. A small model and training loop

The model is a 2-conv-layer CNN with a dropout-regularized classifier head. Nothing
fancy — the point of the lab is the *tracking*, not the model.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, dropout: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def load_data(batch_size: int, train_size: int = 6000, val_size: int = 1500):
    tfm = T.Compose([T.ToTensor(), T.Normalize((0.286,), (0.353,))])
    root = "./data"
    train = torchvision.datasets.FashionMNIST(root, train=True, download=True, transform=tfm)
    test  = torchvision.datasets.FashionMNIST(root, train=False, download=True, transform=tfm)
    # Subsample to keep the lab fast — full dataset works the same way
    rng = np.random.default_rng(0)
    tr_idx = rng.choice(len(train), train_size, replace=False)
    va_idx = rng.choice(len(test),  val_size,   replace=False)
    tr = DataLoader(Subset(train, tr_idx.tolist()), batch_size=batch_size, shuffle=True)
    va = DataLoader(Subset(test,  va_idx.tolist()), batch_size=512)
    return tr, va


### The instrumented training loop

The pattern: open an `mlflow.start_run()` context, log the params at the top, log
metrics inside the epoch loop with a step number, and log the final artefacts (best
checkpoint + a small summary) at the end.

In [ ]:
def train_one(lr: float, weight_decay: float, batch_size: int,
              dropout: float, epochs: int = 5, run_name: str | None = None):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(dict(lr=lr, weight_decay=weight_decay,
                               batch_size=batch_size, dropout=dropout,
                               epochs=epochs, model="SmallCNN"))

        tr, va = load_data(batch_size)
        model = SmallCNN(dropout=dropout).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

        best_val = float("inf")
        best_state = None
        t0 = time.time()
        for ep in range(epochs):
            # ---- train
            model.train()
            tr_loss_sum, n = 0.0, 0
            for x, y in tr:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                loss = F.cross_entropy(model(x), y)
                loss.backward(); opt.step()
                tr_loss_sum += loss.item() * x.size(0); n += x.size(0)
            tr_loss = tr_loss_sum / n

            # ---- validate
            model.eval()
            va_loss_sum, correct, n = 0.0, 0, 0
            with torch.no_grad():
                for x, y in va:
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    logits = model(x)
                    va_loss_sum += F.cross_entropy(logits, y, reduction="sum").item()
                    correct += (logits.argmax(1) == y).sum().item()
                    n += x.size(0)
            va_loss, va_acc = va_loss_sum / n, correct / n

            mlflow.log_metric("train/loss", tr_loss, step=ep)
            mlflow.log_metric("val/loss",   va_loss, step=ep)
            mlflow.log_metric("val/acc",    va_acc,  step=ep)

            if va_loss < best_val:
                best_val = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        # ---- end-of-run summary + checkpoint
        wall = time.time() - t0
        mlflow.log_metric("best/val_loss", best_val)
        mlflow.log_metric("wall_seconds", wall)

        ckpt_path = Path(f"./best_{run.info.run_id[:8]}.pt")
        torch.save(best_state, ckpt_path)
        mlflow.log_artifact(str(ckpt_path), artifact_path="checkpoint")
        ckpt_path.unlink()

        return dict(run_id=run.info.run_id, best_val=best_val, wall=wall)


# Smoke test — one run with a reasonable default
_ = train_one(lr=1e-3, weight_decay=1e-4, batch_size=128, dropout=0.3,
              epochs=3, run_name="smoke-test")


**What just happened.** Look in `./mlruns` — there is one experiment folder with
one run subfolder, containing the params, metrics, and artefacts we logged. You can
also browse this in a UI by running `mlflow ui` from the repo root.

In the rest of the lab we do not need the UI — we'll pull the data out programmatically
and produce the standard sweep plots ourselves.

## 2. Optuna sweep

20 trials, TPE sampler. Each trial is a `train_one(...)` call wrapped in an Optuna
objective. Optuna keeps its own database of trials; we also keep MLflow logging the
same trials, so we have both views.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 5e-2, log=True)
    wd = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    bs = trial.suggest_categorical("batch_size", [64, 128, 256])
    do = trial.suggest_float("dropout", 0.0, 0.5)
    info = train_one(lr=lr, weight_decay=wd, batch_size=bs, dropout=do,
                     epochs=3, run_name=f"trial-{trial.number:02d}")
    trial.set_user_attr("mlflow_run_id", info["run_id"])
    return info["best_val"]


study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=20, show_progress_bar=False)

print(f"Best trial: #{study.best_trial.number}  val_loss={study.best_value:.4f}")
print(f"Best params: {study.best_params}")


## 3. Sweep-analysis plots, offline

We will assemble the sweep results into a single dataframe and produce the four
canonical views: parallel coordinates, parameter importance, metric vs. each
hyperparameter, and a slice grid.

### 3.1 Pull the sweep into a dataframe

In [ ]:
def study_to_df(study) -> pd.DataFrame:
    rows = []
    for t in study.trials:
        if t.state != optuna.trial.TrialState.COMPLETE: continue
        rows.append({**t.params, "val_loss": t.value, "trial": t.number})
    return pd.DataFrame(rows)


sweep = study_to_df(study)
sweep.head()


### 3.2 Parallel coordinates (Matplotlib, no Plotly)

Each row of the dataframe becomes a polyline through the per-axis-normalized
coordinates. Numeric axes are linear after a per-column min-max normalization;
categorical axes are spaced at integer positions.

In [ ]:
def parallel_coordinates(df: pd.DataFrame, axes_cols, metric_col,
                         log_axes=(), figsize=(11, 4.5), cmap="viridis"):
    fig, ax = plt.subplots(figsize=figsize)

    # Normalize each axis to [0, 1]
    norms = {}
    for col in axes_cols:
        v = df[col]
        if v.dtype == object:                                      # categorical
            cats = sorted(v.unique())
            mapping = {c: i / max(len(cats) - 1, 1) for i, c in enumerate(cats)}
            norms[col] = (v.map(mapping).values, cats, "cat")
        else:
            if col in log_axes:
                vals = np.log10(v.values)
            else:
                vals = v.values
            lo, hi = vals.min(), vals.max()
            span = hi - lo if hi > lo else 1.0
            norms[col] = ((vals - lo) / span, (lo, hi), "log" if col in log_axes else "lin")

    metric = df[metric_col].values
    cmap = plt.get_cmap(cmap)
    m_lo, m_hi = metric.min(), metric.max()
    colors = cmap((metric - m_lo) / (m_hi - m_lo + 1e-12))

    xs = np.arange(len(axes_cols))
    for i in range(len(df)):
        ys = [norms[c][0][i] for c in axes_cols]
        ax.plot(xs, ys, color=colors[i], alpha=0.7, lw=1.5)

    # Per-axis tick labels
    ax.set_xticks(xs, axes_cols)
    ax.set_ylim(-0.02, 1.02); ax.set_yticks([])
    # Annotate each axis with its true min/max
    for x, c in zip(xs, axes_cols):
        _, info, kind = norms[c]
        if kind == "cat":
            ax.text(x, 1.04, str(info[-1]), ha="center", va="bottom", fontsize=8)
            ax.text(x, -0.04, str(info[0]), ha="center", va="top", fontsize=8)
        else:
            lo, hi = info
            fmt = (lambda v: f"{10**v:.0e}") if kind == "log" else (lambda v: f"{v:.2f}")
            ax.text(x, 1.04, fmt(hi), ha="center", va="bottom", fontsize=8)
            ax.text(x, -0.04, fmt(lo), ha="center", va="top", fontsize=8)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(m_lo, m_hi))
    fig.colorbar(sm, ax=ax, label=metric_col, pad=0.02)
    ax.set_title(f"Parallel coordinates — colour = {metric_col}")
    plt.tight_layout()
    return fig, ax


parallel_coordinates(sweep, ["lr", "weight_decay", "batch_size", "dropout"],
                     "val_loss", log_axes=["lr", "weight_decay"])
plt.show()


**Reading.** Trace each line from left to right. Dark (low-loss) lines tend to
pass through certain regions of the axes — that's where the productive part of the
hyperparameter space lives. For our small CNN, low-loss runs tend to have moderate `lr`
($10^{-3}$ region), small-to-medium `weight_decay`, and modest dropout. Lines that
cross sharply between adjacent axes flag *interactions* worth investigating.

### 3.3 Parameter importance

Optuna exposes a built-in importance estimator (functional ANOVA). It is sensitive to
the trial count, so treat it as a rough guide.

In [ ]:
importances = optuna.importance.get_param_importances(study)
imp = pd.Series(importances).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(imp.index, imp.values, color="#0072B2")
ax.set(xlabel="importance (functional ANOVA)",
       title="Parameter importance — 20 trials")
plt.show()
imp.round(3)


### 3.4 Metric vs. each hyperparameter (slice plot)

In [ ]:
params = ["lr", "weight_decay", "batch_size", "dropout"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, p in zip(axes, params):
    if p in ("lr", "weight_decay"):
        ax.set_xscale("log")
    sub = sweep.sort_values(p)
    ax.scatter(sub[p], sub["val_loss"], s=36, c=sub["val_loss"],
               cmap="viridis_r", edgecolor="white")
    ax.set(xlabel=p, ylabel="val_loss" if ax is axes[0] else "",
           title=p)
plt.tight_layout(); plt.show()


**Reading.** `lr` shows the clearest pattern — a U-shape with the minimum near
$10^{-3}$. `dropout` shows a milder trend. `weight_decay` and `batch_size` look mostly
flat, which agrees with the importance bars above. **This is the chart that tells you
to stop sweeping `batch_size` and start sweeping `lr` more densely.**

## 4. Loading a checkpoint from a run id

The last piece: prove that the artefacts are usable. We pick the best run's id from
the study, download its checkpoint from the MLflow store, and verify it gives the same
val accuracy on a fresh DataLoader.

In [ ]:
best_run_id = study.best_trial.user_attrs["mlflow_run_id"]
client = mlflow.MlflowClient()
best_run = client.get_run(best_run_id)
print("Best run params:", best_run.data.params)
print("Best run metrics:", {k: round(v, 4) for k, v in best_run.data.metrics.items()})

local_ckpt = mlflow.artifacts.download_artifacts(
    run_id=best_run_id, artifact_path="checkpoint",
)
ckpt_file = next(Path(local_ckpt).glob("*.pt"))
print("Downloaded checkpoint:", ckpt_file)

# Reload and re-validate
dropout = float(best_run.data.params["dropout"])
model = SmallCNN(dropout=dropout).to(DEVICE)
model.load_state_dict(torch.load(ckpt_file, map_location=DEVICE))
model.eval()

_, va = load_data(batch_size=int(best_run.data.params["batch_size"]))
correct, n = 0, 0
with torch.no_grad():
    for x, y in va:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        n += x.size(0)
print(f"Reloaded model val acc: {correct / n:.4f}")


## What to do differently in your own research

- Instrument **every** training run, including the throwaway ones. The cost is one
  function decorator; the benefit is being able to answer "why did you try that
  learning rate?" three months later.
- Log enough that a chart in your paper can be **regenerated from the run id**. Save
  the validation predictions, not just the metrics derived from them.
- Treat 20 random-search runs as a **scouting expedition**, not the final sweep. Use
  the parallel-coordinates plot to redesign the search space and re-launch.
- The parallel-coordinates plot and the parameter-importance bar are **complementary**.
  Always inspect both — they sometimes disagree and the disagreement is interesting.

### Next

Open `exercises/01-sweep.ipynb` — you will build a parallel-coordinates plot from a
*provided* sweep dataframe (no GPU needed for the exercise), and analyze it.
